# project_19_petase — all notebooks (00→05) in one

This is a **convenience copy** that concatenates the six standalone notebooks in order so you can run the whole project top-to-bottom in a single Colab session. The individual notebooks (`00_setup.ipynb` … `05_validation_plan.ipynb`) remain in this folder and are the canonical deliverables. Sections are separated by dividers; each section keeps its own setup/`import` cells (re-running them is harmless). All synthetic numbers are still labeled `EXAMPLE_DATA`.

---

## ▶︎ Section 1 / 6 — `00_setup.ipynb`

---

# 00 · Environment Setup — De Novo Protein Design Capstone

This is the **shared setup notebook** every project starts from. Run it top to bottom
*once per Colab session*. It:

1. detects your GPU and warns if you're on a weak/absent one,
2. installs a light, pinned core toolset (Biopython, py3Dmol, foldseek-less utilities),
3. optionally installs heavier tools (ColabFold, ESMFold) on demand,
4. prints exact versions for your `LOG.md` (reproducibility is graded).

> **Compute reality.** A free Colab **T4** runs ColabFold, ESMFold, ProteinMPNN, and small
> RFdiffusion jobs. **BindCraft / RFantibody / large RFdiffusion** want an **A100** (Colab Pro+
> or a cluster). Each project's `MANUAL.md` states its tier. Don't fight a T4 to do an A100 job —
> plan your batch sizes around it.

## 1 · GPU & environment check

In [ ]:
import subprocess, sys, platform, textwrap

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())

gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
if gpu:
    print("GPU    :", gpu)
    name = gpu.lower()
    if "t4" in name:
        print(textwrap.fill(
            "NOTE: T4 detected. Good for ColabFold/ESMFold/ProteinMPNN/small RFdiffusion. "
            "For BindCraft/RFantibody/large diffusion, switch to A100 (Colab Pro+) or a cluster.", 88))
    elif any(x in name for x in ("a100", "l4", "v100")):
        print("NOTE: capable GPU — heavier tools (BindCraft/RFantibody) are feasible.")
else:
    print("GPU    : NONE FOUND")
    print(textwrap.fill(
        "WARNING: No GPU. Go to Runtime → Change runtime type → Hardware accelerator → GPU. "
        "Structure prediction on CPU is impractically slow.", 88))

## 2 · Pinned core install (fast, T4-friendly)

These are light and used across every project. Pins are conservative; bump them in your repo if needed and **log it**.

In [ ]:
# Core utilities used in every project. Quiet + pinned.
%pip -q install biopython==1.84 py3Dmol==2.4.0 numpy pandas matplotlib seaborn tqdm requests 2>/dev/null
print("Core install done.")

In [ ]:
# Version stamp — copy this block's output into your LOG.md for reproducibility.
import importlib, datetime
mods = ["Bio", "py3Dmol", "numpy", "pandas", "matplotlib", "seaborn", "tqdm", "requests"]
print("# Environment stamp", datetime.datetime.utcnow().isoformat(timespec="seconds"), "UTC")
for m in mods:
    try:
        v = importlib.import_module(m).__version__
    except Exception:
        v = "n/a"
    print(f"{m:14s} {v}")

## 3 · Heavy tools — install *on demand*

Don't install these unless your project needs them this session (they're slow to set up).
Each is wrapped in a function so you only pay the cost when you call it.

In [ ]:
def install_colabfold():
    """ColabFold (AF2). ~3–5 min on first install. T4 OK."""
    import subprocess
    subprocess.run("pip -q install 'colabfold[alphafold-minus-jax]'", shell=True)
    # On Colab, the standard route is the localcolabfold installer or the ColabFold notebook;
    # here we expose the pip route. If it fails, fall back to the official ColabFold notebook
    # and import your sequences. Log whichever path you used.
    print("ColabFold install attempted. Verify with: from colabfold.batch import run")

def install_esmfold():
    """ESMFold via HuggingFace transformers. T4 OK for <~400 aa."""
    import subprocess
    subprocess.run("pip -q install 'transformers>=4.40' accelerate", shell=True)
    print("ESMFold deps installed. Load with transformers EsmForProteinFolding.")

print("Helpers ready: install_colabfold(), install_esmfold().")

## 4 · Reproducibility helpers

Call `set_seeds()` at the top of every run, and use `log()` to append to your `LOG.md`.

In [ ]:
import os, random
import numpy as np

def set_seeds(seed: int = 0):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    print(f"seeds set to {seed}")

def log(msg: str, path: str = "LOG.md"):
    import datetime
    stamp = datetime.datetime.utcnow().isoformat(timespec="seconds")
    with open(path, "a") as fh:
        fh.write(f"- {stamp}Z · {msg}\n")
    print("logged:", msg)

set_seeds(0)
log("Ran 00_setup; environment stamped.")

## 5 · (Optional) Mount Google Drive for persistence

Colab sessions are ephemeral. Mount Drive to keep your `results/` and design pools between sessions.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# WORKDIR = "/content/drive/MyDrive/denovo_capstone/project_XX"
# import os; os.makedirs(WORKDIR, exist_ok=True); os.chdir(WORKDIR)
print("Uncomment to mount Drive and set your working directory.")

---
**Next:** open `01_define_and_explore.ipynb`. Keep this session alive — re-running `00_setup`
each new session is normal. Record every version and seed in `LOG.md`.

---

## ▶︎ Section 2 / 6 — `01_define_and_explore.ipynb`

---

# 01 · Define & Explore — serine-hydrolase/PETase catalysis, the triad, and the thermostability bottleneck

**Standard slot:** *define & explore.* **For Project 19 this means:** understand de novo enzyme
design and PET hydrolysis (a serine-hydrolase reaction), then **construct the theozyme** (the
Ser-His-Asp triad + oxyanion hole around an ester transition state) and run a mock theozyme→scaffold
hello-world (D0). The emphasis throughout is **thermostability** — the real bottleneck for PET.

Run `00_setup.ipynb` first in this session.

## Why PET hydrolases — and why thermostability is the bottleneck
PET (polyethylene terephthalate) is a hugely-produced plastic that is barely recycled. **PET
hydrolases** (IsPETase, the engineered cutinase LCC) cut the PET ester backbone back to its monomers
(MHET / TPA + ethylene glycol), enabling **circular chemistry**. Two facts shape this project:
- **The chemistry is known.** PET hydrolysis is a textbook **serine-hydrolase** reaction — the same
  **Ser-His-Asp triad + oxyanion hole** as lipases, esterases, and cutinases. Unlike the Kemp
  elimination (Project 18), this reaction *has* natural counterparts.
- **Stability is the hard part.** PET only becomes accessible to enzymes near its glass transition
  (~65-70 °C), and **wild-type IsPETase falls apart there.** The breakthrough (Tournier 2020) was
  *thermostabilising* the enzyme. So this campaign is decided by **thermostability**, not catalytic
  novelty.

The honest history: de novo enzymes are rarely active first try (<5% without directed evolution), and
even when geometry is right, the thermostability ↔ activity trade-off bites.

## The theozyme — the catalytic motif you must build
A **theozyme** ("theoretical enzyme") is the minimal set of catalytic functional groups placed
around the **transition state** (here, the tetrahedral intermediate of ester hydrolysis):

| Role | Residue(s) | Job in the TS |
|------|-----------|----------------|
| catalytic Ser | Ser (OG) | nucleophile — attacks the ester carbonyl carbon |
| catalytic His | His (NE2) | general base — deprotonates Ser-OH to activate it |
| catalytic Asp | Asp / Glu (OD) | orients/protonates His (the charge-relay) |
| oxyanion hole | 2× backbone-NH (or Ser-OG) | stabilise the developing oxyanion of the tetrahedral intermediate |

The **oxyanion hole is easy to forget and decisive** — without it the tetrahedral intermediate is not
stabilised and there is no catalysis even with a perfect triad. You **construct** this from the
literature and/or a QM transition-state model — it is a teaching template
(`data/inputs/theozyme_def.txt`), **not** fabricated data. Place groups around the **TS**, not the
ground-state ester.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Build the theozyme (mock hello-world)
`scripts/enzyme_tools.py` exposes `build_theozyme(reaction)` → a functional-group geometry spec.
The distances/angles it ships are **PLACEHOLDERS** — replace them in `data/inputs/theozyme_def.txt`
(and in `build_theozyme`) with real, cited values during P1. This follows the **enzyme-family
template** (Project 18, Kemp): the reaction (ester hydrolysis) and emphasis (thermostability) differ;
the workflow is the same.

In [ ]:
from enzyme_tools import build_theozyme

theo = build_theozyme("ester_hydrolysis")
print("Reaction :", theo.reaction)
print("Substrate:", theo.substrate)
print("Provenance:", theo.provenance)
print("\nCatalytic functional groups (PLACEHOLDER geometry — fill from literature/QM):")
for fg in theo.functional_groups:
    ang = f"{fg.target_angle}deg" if fg.target_angle is not None else "n/a"
    print(f"  {fg.role:16s} {fg.residue}/{fg.atom:4s}  d={fg.target_distance}A  angle={ang}")
print("\nCatalytic residues to FIX during sequence design (triad + oxyanion hole):")
print(" ", theo.catalytic_residue_ids())

## A first mock scaffold + sequence + thermostability proxy (no GPU)
`scaffold_motif(...)` (mock) returns placeholder backbones presenting the motif;
`ligandmpnn_fix_catalytic(...)` (mock) designs sequences with the catalytic triad fixed;
`thermostability_md(...)` (mock) returns the project's headline stability proxy. **Every number here
is SYNTHETIC** — this only proves the plumbing runs anywhere. Switch to the real backends
(RFdiffusion2/Riff-Diff + OpenMM on an A100; LigandMPNN CPU-fast) in `02_generate.ipynb` / `04`.

In [ ]:
from enzyme_tools import (scaffold_motif, ligandmpnn_fix_catalytic,
                          catalytic_geometry_rmsd, thermostability_md, dock_substrate)

scaffolds = scaffold_motif(theo, n=5, method="mock", track="de_novo")
print(f"{len(scaffolds)} mock scaffolds; example:")
print(" ", scaffolds[0])

seqs = ligandmpnn_fix_catalytic(scaffolds[0], theo.catalytic_residue_ids(), n=3, tool="mock")
print(f"\n{len(seqs)} mock sequences for {scaffolds[0]['design_id']} "
      f"(fixed roles: {seqs[0]['fixed_catalytic_roles']})")

cg = catalytic_geometry_rmsd(None, theo)         # mock, SYNTHETIC
thermo = thermostability_md(None, ns=20.0)       # mock, SYNTHETIC — the project emphasis
dock = dock_substrate(None, theo.substrate)      # mock, SYNTHETIC — pocket accessibility
print(f"\ncatalytic_geometry_rmsd (mock, SYNTHETIC) = {cg} A  -> pass if < 0.5 A")
print(f"thermostability proxy (mock, SYNTHETIC): catalytic_rmsf={thermo['catalytic_rmsf']} A, "
      f"melting_proxy={thermo['melting_proxy']} (higher=more stable)")
print(f"PET-mimic docking (mock, SYNTHETIC): in_pocket={dock['pose_in_pocket']}, "
      f"oriented_to_Ser={dock['oriented_to_ser']}")
print("\nNOTE: these are placeholder numbers. The real campaign is in notebook 02; the "
      "thermostability ranking is in notebook 04.")

## The metrics that decide a PET-hydrolase design
| Metric | Cutoff (`enzyme`) | Means | Does **not** mean |
|--------|-------------------|-------|-------------------|
| scRMSD | ≤ 2.0 Å | designed-vs-predicted backbone self-consistency | activity |
| pLDDT (global) | ≥ 85 | local fold confidence | thermostability / catalysis |
| pLDDT (catalytic) | ≥ 90 | confidence *at the triad + oxyanion hole* | the geometry is correct |
| **catalytic_geom_rmsd** | **< 0.5 Å** | predicted catalytic atoms vs the theozyme | **activity** (the design can still be dead) |
| **thermostability proxy** (MD) | *rank, not a hard cutoff* | RMSF + melting-proxy — survives heat? | **a measured Tm** (DSF decides) |

The last two rows are the point of this project — and the last column is the message to never forget:
**in-silico catalytic geometry + an MD stability proxy do not guarantee a working, thermostable
enzyme.** Only an activity assay (pNP-ester / PET-film) + DSF decide (notebook 05).

## D0 checklist
- [ ] Half-page on de novo enzyme design + the honest hit-rate history + **why thermostability is the PET bottleneck**.
- [ ] 1-page problem statement with **measurable** success criteria (incl. a thermostability target) + the controls you'll need.
- [ ] Theozyme spec started in `data/inputs/theozyme_def.txt` (triad + oxyanion hole; replace the PLACEHOLDERs, cite sources).
- [ ] Reproduced mock hello-world (triad + oxyanion-hole spec + a mock scaffold record + a thermostability proxy).
- [ ] `LOG.md` entry (tool versions, GPU, seed).

**Next:** `02_generate.ipynb` — scaffold the triad into stable folds and run LigandMPNN with the catalytic triad fixed.

---

## ▶︎ Section 3 / 6 — `02_generate.ipynb`

---

# 02 · Campaign — theozyme → scaffold (stable folds) → LigandMPNN (triad fixed)

**Standard slot:** *design campaign.* **For Project 19 this means:** take the triad + oxyanion-hole
theozyme, scaffold it into many **thermostable** backbones (RFdiffusion2 / Riff-Diff — the **A100**
step), then **LigandMPNN sequence design fixing the catalytic triad**, and write a results CSV (D2).
Optionally run two tracks — **fully de novo** vs **engineered-natural** (graft onto a stable
cutinase/IsPETase scaffold) — to compare in notebook 04.

Runs end-to-end on the **mock** backend with no GPU; switch to the real backends on Colab/HPC.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams
Tools change. Before a campaign, HTTP-check that the pinned upstream repos still exist, and pin the
commit/tag you actually use. **RFdiffusion2 and Riff-Diff are new and move fast — VERIFY the current
public release/repo at generation time** (do not assert a repo you are unsure of); the others below
are stable enough to head-check.

In [ ]:
import requests

# Pinned upstreams (pin the COMMIT/TAG you use in env/requirements.txt + LOG.md):
STABLE_UPSTREAMS = {
    "RFdiffusion (classic motif scaffolding)": "https://github.com/RosettaCommons/RFdiffusion",
    "LigandMPNN (catalytic-triad-fixed seq design)": "https://github.com/dauparas/LigandMPNN",
    "AutoDock Vina (PET-mimic ester fit)": "https://github.com/ccsb-scripps/AutoDock-Vina",
    "OpenMM (thermostability MD — the project emphasis)": "https://github.com/openmm/openmm",
}
# VERIFY-ONLY (new/fast-moving; confirm the current release before relying on a URL):
VERIFY_UPSTREAMS = [
    "RFdiffusion2 (Dauparas 2025) — VERIFY current public release/repo at generation time",
    "Riff-Diff (Schnettler 2025, Nature) — VERIFY current public release/repo at generation time",
]

for name, url in STABLE_UPSTREAMS.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=15)
        print(f"[{r.status_code}] {name}\n      {url}")
    except Exception as e:
        print(f"[ERR] {name}: {e!r}\n      {url}")
print("\nVERIFY MANUALLY (do not assert a repo URL you are unsure of):")
for v in VERIFY_UPSTREAMS:
    print("  -", v)

## 1 · Build the theozyme and scaffold it (into stable folds)
The mock path returns placeholder backbones so the loop runs anywhere. On an A100, switch
`METHOD` to `"rfdiffusion2"` or `"riffdiff"` (verify the release) and `N_SCAFFOLDS` to 1000s, biasing
toward compact α/β-hydrolase-like (thermostable) topologies. We tag a `TRACK` so notebook 04 can
compare **fully de novo** vs **engineered-natural** scaffolds.

> **A100 NOTE:** scaffolding 1000s of backbones is the compute bottleneck. Free Colab T4 can do a
> small **RFdiffusion** (classic) motif-scaffolding demo (tens of backbones); the real campaign wants
> an A100 (Colab Pro+) or HPC. The mock backend below needs no GPU at all.

In [ ]:
from enzyme_tools import build_theozyme, scaffold_motif

theo = build_theozyme("ester_hydrolysis")

METHOD = "mock"        # -> "rfdiffusion2" | "riffdiff" | "rfdiffusion" on Colab/HPC (verify release)
N_SCAFFOLDS = 12       # -> 1000s for the real campaign
TRACK = "de_novo"      # -> also run "engineered_natural" (graft onto a stable cutinase/IsPETase) for the nb04 comparison

scaffolds = scaffold_motif(theo, n=N_SCAFFOLDS, method=METHOD, track=TRACK)
print(f"{len(scaffolds)} scaffolds via method={METHOD!r} track={TRACK!r} (mock numbers are SYNTHETIC)")
print("example:", scaffolds[0])

## 2 · LigandMPNN sequence design — FIXING the catalytic triad + oxyanion hole
This is the core of enzyme sequence design: redesign the protein but **keep the triad AND the
oxyanion-hole donors fixed** (and use the ligand/ester-TS context). That is why LigandMPNN, not
vanilla ProteinMPNN, is used. On Colab set `TOOL="ligandmpnn"` (CPU-fast) and pass the fixed-positions
list + ligand/TS context. You MAY bias the (non-catalytic) surface toward thermostabilising residues —
never touch the triad.

In [ ]:
from enzyme_tools import ligandmpnn_fix_catalytic

TOOL = "mock"          # -> "ligandmpnn" on Colab (CPU-fast)
SEQS_PER_BACKBONE = 4

catalytic = theo.catalytic_residue_ids()   # triad + oxyanion hole — ALL fixed
all_designs = []
for bb in scaffolds:
    seqs = ligandmpnn_fix_catalytic(bb, catalytic, n=SEQS_PER_BACKBONE, tool=TOOL)
    for s in seqs:
        s["scaffold_id"] = bb["design_id"]
        s["scaffold_method"] = bb["method"]
        s["track"] = bb["track"]
        s["motif_rmsd"] = bb["motif_rmsd"]
        all_designs.append(s)
print(f"{len(all_designs)} sequences total "
      f"({len(scaffolds)} backbones x {SEQS_PER_BACKBONE}); catalytic roles fixed: {catalytic}")

## 3 · Predict + score (mock geometry + thermostability), write the results CSV
On Colab, predict each sequence with AF2/ESMFold, read the **active-site pLDDT**, compute the real
`catalytic_geometry_rmsd` from the predicted PDB, run the **thermostability MD** (RMSF + melting-proxy),
and dock the PET-mimic ester. Here the mock backend fills SYNTHETIC values so the CSV — the input to
notebook 03 — is produced anywhere. (No kcat, no Tm is fabricated.)

In [ ]:
import pandas as pd
from enzyme_tools import catalytic_geometry_rmsd, dock_substrate, thermostability_md

rows = []
for d in all_designs:
    # On Colab, `pred_pdb` is the AF2-predicted PDB path for this design; the mock backend keys off
    # the (non-existent) per-design path string so each design gets a DISTINCT SYNTHETIC value.
    pred_pdb = f"results/pred/{d['design_id']}.pdb"
    cg = catalytic_geometry_rmsd(pred_pdb, theo)            # mock -> SYNTHETIC (varies per design)
    dock = dock_substrate(pred_pdb, theo.substrate)         # mock -> SYNTHETIC (pocket accessibility)
    thermo = thermostability_md(pred_pdb, ns=20.0)          # mock -> SYNTHETIC (the project emphasis)
    # SYNTHETIC stand-ins for AF2 confidence so the plumbing runs (replace with real predictions):
    import hashlib
    h = int(hashlib.sha256(d["design_id"].encode()).hexdigest(), 16)
    plddt = 78 + (h % 20)            # 78-97, SYNTHETIC
    plddt_cat = 80 + ((h >> 7) % 18) # 80-97, SYNTHETIC
    scrmsd = round(0.8 + ((h >> 11) % 200) / 100.0, 2)  # 0.8-2.8, SYNTHETIC
    rows.append(dict(
        design_id=d["design_id"], scaffold_id=d["scaffold_id"],
        scaffold_method=d["scaffold_method"], track=d["track"], sequence=d["sequence"],
        plddt=plddt, plddt_catalytic=plddt_cat, scrmsd=scrmsd,
        catalytic_geom_rmsd=cg, vina_score=dock["vina_score"],
        pose_in_pocket=dock["pose_in_pocket"], oriented_to_ser=dock["oriented_to_ser"],
        catalytic_rmsf=thermo["catalytic_rmsf"], backbone_rmsf=thermo["backbone_rmsf"],
        melting_proxy=thermo["melting_proxy"],
        thermostability_rank_score=thermo["thermostability_rank_score"],
        md_rmsd=thermo["backbone_rmsf"],  # reuse backbone fluctuation as the dynamics-layer RMSD proxy
        synthetic=True))

camp = pd.DataFrame(rows)
camp.to_csv("results/campaign.csv", index=False)
print("wrote results/campaign.csv", camp.shape, "(ALL NUMBERS SYNTHETIC — mock backend)")
print(f"catalytic_geom_rmsd range: {camp['catalytic_geom_rmsd'].min()}-{camp['catalytic_geom_rmsd'].max()} A")
print(f"melting_proxy range: {camp['melting_proxy'].min()}-{camp['melting_proxy'].max()} (higher=more stable)")
camp.head()

## D2 checklist
- [ ] Scaffolding run logged (method, release/commit, N backbones, track, seed) — A100 for the real campaign.
- [ ] LigandMPNN sequences with the **catalytic triad + oxyanion hole provably fixed** (fixed-positions list logged).
- [ ] `results/campaign.csv` with one row per design (real metrics on Colab; mock here), incl. a thermostability proxy.
- [ ] Design log (every config + seed + output path) + 3–4 page interim report.
- [ ] *(extension)* a second `track="engineered_natural"` run for the nb04 comparison.

**Next:** `03_filter_and_rank.ipynb` — run the shared enzyme filter on `campaign.csv`.

---

## ▶︎ Section 4 / 6 — `03_filter_and_rank.ipynb`

---

# 03 · Filter & Rank — the shared enzyme filter

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25
projects use. **For Project 19** you filter on the **enzyme** cutoffs, with the catalytic-geometry
RMSD as the decisive geometry metric. (The **thermostability ranking** — this project's emphasis —
is applied to the survivors in notebook 04.)

Run `00`–`02` first so `results/campaign.csv` exists.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline
Improvements here are pull-requested back to `shared/` for the whole cohort — do not silently fork it.

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("enzyme cutoffs:", fp.DEFAULT_CUTOFFS["enzyme"])

## Build `fp.Design` objects (enzyme) from the campaign
Map each campaign row onto an `fp.Design`, carrying the enzyme-specific fields: `plddt`,
**`plddt_catalytic`**, `scrmsd`, and **`catalytic_geom_rmsd`**. The self-consistency layer
(`self_consistency`) checks all of these against the enzyme cutoffs (scrmsd ≤ 2.0, plddt ≥ 85,
plddt_cat ≥ 90, cat_geom ≤ 0.5). We carry the thermostability proxy in `extra` for the nb04 ranking,
and use `md_rmsd` (backbone fluctuation) for the optional dynamics layer.

In [ ]:
camp = pd.read_csv("results/campaign.csv")

designs = []
for _, r in camp.iterrows():
    designs.append(fp.Design(
        design_id=str(r["design_id"]),
        sequence=str(r.get("sequence", "")),
        design_type="enzyme",
        plddt=float(r["plddt"]),
        plddt_catalytic=float(r["plddt_catalytic"]),
        scrmsd=float(r["scrmsd"]),
        catalytic_geom_rmsd=float(r["catalytic_geom_rmsd"]),
        md_rmsd=float(r["md_rmsd"]),
        extra={"scaffold_method": r["scaffold_method"], "track": r["track"],
               "melting_proxy": float(r["melting_proxy"]),
               "thermostability_rank_score": float(r["thermostability_rank_score"]),
               "synthetic": True},
    ))
print(len(designs), "enzyme Design objects built (from SYNTHETIC mock metrics)")

## Run the pipeline (`design_type="enzyme"`) and report
`run_pipeline` applies the layers in order and returns a ranked DataFrame. We use layers 1+3+4
(self-consistency incl. catalytic geometry, physics, and the short-MD dynamics layer); the orthogonal
layer (L2) needs a second predictor's scRMSD, which you add on Colab. The **thermostability ranking**
beyond these layers is in notebook 04.

In [ ]:
df_ranked = fp.run_pipeline(designs, design_type="enzyme", use_layers=(1, 3, 4))
df_ranked.to_csv("results/ranked.csv", index=False)
top = fp.report(df_ranked, top_n=10, save_prefix="results/proj19")
top

## Survival-at-each-layer + catalytic-geometry pass rate (honest accounting)
The **catalytic-geometry layer is where most enzyme designs die** — expect the steepest drop there.
Report the pass rate explicitly; this is a headline benchmark for D3 (the other is the
thermostability ranking in nb04).

In [ ]:
import pandas as pd
print("layers_passed distribution:")
print(df_ranked["layers_passed"].value_counts().sort_index())

n = len(df_ranked)
cut = fp.DEFAULT_CUTOFFS["enzyme"]["cat_geom"]
n_geom = int((df_ranked["catalytic_geom_rmsd"] <= cut).sum())
print(f"\ncatalytic-geometry preservation: {n_geom}/{n} "
      f"({100*n_geom/max(n,1):.1f}%) hold the motif < {cut} A  [SYNTHETIC demo numbers]")
print("Reminder: geometry pass != activity, and != thermostability. nb04 ranks survivors by the "
      "MD thermostability proxy; nb05 plans the activity + DSF assays.")

## D3 (part 1) checklist
- [ ] `results/ranked.csv` produced by the **shared** module with `design_type="enzyme"`.
- [ ] Survival-at-each-layer figure (`results/proj19_survival.png`).
- [ ] Catalytic-geometry preservation rate reported (a headline metric).
- [ ] Mapping assumptions written down (which metrics, which prediction); thermostability proxy carried in `extra`.

**Next:** `04_validate.ipynb` — catalytic-geometry preservation + **MD-thermostability ranking** + pocket docking + engineered-vs-de-novo comparison.

---

## ▶︎ Section 5 / 6 — `04_validate.ipynb`

---

# 04 · Validate — geometry preservation, MD-THERMOSTABILITY ranking, docking & track comparison

**Standard slot:** *validate (in silico).* **For Project 19 the benchmark has two headlines:** the
**catalytic-geometry preservation rate** and the **MD-based thermostability ranking** (RMSF +
melting-proxy) — the project's emphasis. Plus **PET-mimic pocket accessibility** (docking) and the
**engineered-natural vs fully de novo** scaffold comparison `[extension]` (D3 pt2).

Needs `results/campaign.csv` (+ `results/ranked.csv` from notebook 03).

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Catalytic-geometry preservation — the geometry headline
Distribution of catalytic-geometry RMSD vs the 0.5 Å pass bar. The fraction left of the line is the
**preservation rate**. (Numbers here are SYNTHETIC mock values; on Colab they come from real AF2
predictions.)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

camp = pd.read_csv("results/campaign.csv")
cut = 0.5

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.hist(camp["catalytic_geom_rmsd"], bins=20)
ax.axvline(cut, color="k", ls="--", lw=1, label=f"pass < {cut} A")
ax.set_xlabel("catalytic-geometry RMSD vs theozyme (A)  [SYNTHETIC]")
ax.set_ylabel("designs"); ax.set_title("Catalytic-geometry preservation (triad + oxyanion hole)")
ax.legend(); plt.tight_layout()
plt.savefig("results/catalytic_geometry_hist.png", dpi=150); plt.show()

rate = 100 * (camp["catalytic_geom_rmsd"] <= cut).mean()
print(f"overall catalytic-geometry preservation rate = {rate:.1f}%  [SYNTHETIC demo]")

## 2 · MD-based thermostability ranking — THE PROJECT EMPHASIS
Rank the **geometry-passing** survivors by the MD thermostability proxy: **low catalytic/backbone
RMSF + a high melting-proxy** ⇒ more likely to survive industrial temperatures (~65-70 °C). This is a
**proxy, not a Tm** — DSF (notebook 05) gives the real number. Mind the **thermostability ↔ activity
trade-off**: do not pick only the most rigid designs.

In [ ]:
geom_ok = camp[camp["catalytic_geom_rmsd"] <= 0.5].copy()
ranked_thermo = geom_ok.sort_values("thermostability_rank_score", ascending=False)
ranked_thermo.to_csv("results/thermostability_ranked.csv", index=False)

fig, ax = plt.subplots(figsize=(5.6, 3.4))
sc = ax.scatter(ranked_thermo["catalytic_rmsf"], ranked_thermo["melting_proxy"],
                c=ranked_thermo["catalytic_geom_rmsd"], cmap="viridis")
ax.set_xlabel("active-site (catalytic) RMSF (A) — lower = more rigid  [SYNTHETIC]")
ax.set_ylabel("melting-proxy (0-1) — higher = more thermostable  [SYNTHETIC]")
ax.set_title("Thermostability ranking of geometry-passing designs")
fig.colorbar(sc, label="catalytic-geom RMSD (A)")
plt.tight_layout(); plt.savefig("results/thermostability_ranking.png", dpi=150); plt.show()

print(f"{len(ranked_thermo)} geometry-passing designs ranked by thermostability [SYNTHETIC].")
print("Top-left + high (rigid active site, high melting-proxy) = most thermostable; but beware the")
print("thermostability <-> activity trade-off — a too-rigid active site can be catalytically dead.")
print("\nTop 5 by thermostability proxy:")
print(ranked_thermo[["design_id", "track", "catalytic_geom_rmsd", "catalytic_rmsf",
                     "melting_proxy", "thermostability_rank_score"]].head().to_string(index=False))

## 3 · Engineered-natural vs fully de novo scaffolds `[extension]`
Compare the two tracks: which holds the triad geometry *and* stays thermostable? In the real campaign
the engineered-natural track grafts the triad onto a stable cutinase/IsPETase scaffold; the de novo
track builds the fold from scratch. Here both tracks may be present if you ran them in nb02; otherwise
this cell shows the *shape* of the comparison you will populate on Colab.

In [ ]:
by_track = (camp.assign(pass_geom=camp["catalytic_geom_rmsd"] <= 0.5)
                .groupby("track")
                .agg(n=("design_id", "size"),
                     geom_pass_rate=("pass_geom", "mean"),
                     mean_melting_proxy=("melting_proxy", "mean"),
                     mean_catalytic_rmsf=("catalytic_rmsf", "mean"),
                     mean_plddt_cat=("plddt_catalytic", "mean"))
                .reset_index())
by_track["geom_pass_rate"] = (100 * by_track["geom_pass_rate"]).round(1)
print("Track comparison (engineered-natural vs fully de novo) [SYNTHETIC]:")
print(by_track.to_string(index=False))
print("\n[SYNTHETIC] On Colab: run BOTH tracks on the SAME theozyme and compare geometry-preservation")
print("AND thermostability — the engineered-natural track often wins on stability, the de novo track")
print("on novelty. Report the trade-off honestly.")

## 4 · PET-mimic substrate docking + active-site stability (top candidates)
Docking (AutoDock Vina) checks the **PET-mimic ester fits and is oriented** toward Ser-OG — pocket
accessibility, not affinity, not activity. Cross it against the thermostability proxy for the ranked
survivors as orthogonal evidence.

In [ ]:
# merge ranked.csv (fp.Design fields) with campaign.csv docking/thermo fields by design_id
try:
    ranked = pd.read_csv("results/ranked.csv")
    ranked = ranked.merge(
        camp[["design_id", "vina_score", "pose_in_pocket", "oriented_to_ser",
              "melting_proxy", "catalytic_rmsf"]],
        on="design_id", how="left")
except FileNotFoundError:
    ranked = camp.copy()

top = ranked.head(min(20, len(ranked)))
fig, ax = plt.subplots(figsize=(5.4, 3.4))
sc = ax.scatter(top["vina_score"], top["melting_proxy"],
                c=top["catalytic_rmsf"], cmap="plasma")
ax.set_xlabel("Vina PET-mimic fit score (more negative = better fit)  [SYNTHETIC]")
ax.set_ylabel("melting-proxy (higher = more thermostable)  [SYNTHETIC]")
ax.set_title("Top candidates: pocket fit vs thermostability")
fig.colorbar(sc, label="catalytic RMSF (A)")
plt.tight_layout(); plt.savefig("results/docking_thermostability.png", dpi=150); plt.show()
print("Lower-left-to-upper (good fit + high melting-proxy + low RMSF) are the best candidates [SYNTHETIC].")

## 5 · Honest hit-rate accounting
Report N(pass all layers) / N(generated), and remind the reader of the field reality: even a good
preservation rate + a good thermostability proxy is **not** an activity rate or a measured Tm.
Geometry ≠ catalysis; an MD proxy ≠ thermostability; activity (pNP-ester / PET-film) + DSF are required.

In [ ]:
n_total = len(camp)
try:
    ranked = pd.read_csv("results/ranked.csv")
    n_hits = int((ranked["layers_passed"] >= 3).sum())
except Exception:
    n_hits = int((camp["catalytic_geom_rmsd"] <= 0.5).sum())
n_thermo = int(((camp["catalytic_geom_rmsd"] <= 0.5) & (camp["melting_proxy"] >= 0.6)).sum())
print("Hit-rate accounting [SYNTHETIC demo]:")
print(f"  generated                          : {n_total}")
print(f"  pass all filter layers             : {n_hits}  ({100*n_hits/max(n_total,1):.1f}%)")
print(f"  geometry-pass AND thermostable proxy: {n_thermo}  ({100*n_thermo/max(n_total,1):.1f}%)")
print("\nREALITY CHECK: de novo enzyme activity rates are <5% without directed evolution; in-silico")
print("catalytic geometry does NOT guarantee activity, and an MD proxy does NOT guarantee real")
print("thermostability. Only an activity assay + DSF decide. Mind the thermostability<->activity trade-off.")

## D3 (part 2) checklist
- [ ] Catalytic-geometry preservation histogram (`results/catalytic_geometry_hist.png`) + rate.
- [ ] **MD-thermostability ranking** figure (`results/thermostability_ranking.png`) + `thermostability_ranked.csv`.
- [ ] Engineered-natural vs fully de novo track comparison `[extension]`.
- [ ] PET-mimic docking + thermostability figure on the ranked top set.
- [ ] Honest hit-rate accounting with the "geometry ≠ activity" + "MD proxy ≠ Tm" caveats stated.

**Next:** `05_validation_plan.ipynb` — the activity + DSF assay plan + controls + surface-redesign stretch.

---

## ▶︎ Section 6 / 6 — `05_validation_plan.ipynb`

---

# 05 · Validation plan — activity assay, DSF thermostability, controls, solubility redesign

**Standard slot:** *validation plan.* **For Project 19 this means:** turn the catalytic-geometry-
filtered, **thermostability-ranked** set into a costed **activity-assay plan** (pNP-ester colorimetric
→ PET-film/HPLC) + a **DSF thermostability** plan with the right controls (incl. a catalytic-Ser→Ala
dead mutant) and a **surface-residue redesign for solubility** stretch (D4/D5).

This is the deliverable that states, plainly: **in-silico geometry + an MD proxy are hypotheses; the
activity assay and DSF test them.**

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Select the synthesis set (< 96 designs)
Pick the top designs from the thermostability-ranked, geometry-passing set, capped at **< 96** so they
fit a single screening plate with controls. **Balance the thermostability ↔ activity trade-off** —
don't pick only the most rigid designs — and keep diversity across scaffolds/tracks.

In [ ]:
import pandas as pd
# Prefer the thermostability-ranked, geometry-passing set from nb04; fall back gracefully.
for src in ("results/thermostability_ranked.csv", "results/ranked.csv", "results/campaign.csv"):
    try:
        ranked = pd.read_csv(src); used = src; break
    except FileNotFoundError:
        continue
print("using:", used)

ok = ranked[ranked["catalytic_geom_rmsd"] <= 0.5] if "catalytic_geom_rmsd" in ranked else ranked
if "thermostability_rank_score" in ok:
    ok = ok.sort_values("thermostability_rank_score", ascending=False)
selected = ok.head(88).copy()
selected.to_csv("results/synthesis_set.csv", index=False)
print(f"selected {len(selected)} designs for synthesis (< 96, leaving wells for controls) [SYNTHETIC ranking]")
print("Diversify across scaffolds/tracks; include a few high-activity-leaning (less rigid) designs to")
print("probe the thermostability<->activity trade-off; record why each was chosen in your report.")

## 2 · The activity assay (pNP-ester fast screen → PET-film/HPLC true substrate)
- **Express + purify:** *E. coli* BL21(DE3) (or SHuffle for disulfide-containing cutinase-like folds),
  16–18 °C overnight; His-tag → IMAC → SEC polish.
- **Fast screen — pNP-ester colorimetric:** p-nitrophenyl acetate/butyrate; follow released
  **p-nitrophenolate at ~405–410 nm** in a plate reader; take initial rates. (A soluble-ester proxy
  for throughput — *not* PET itself.)
- **True substrate — PET-film / amorphous-PET digestion + HPLC:** incubate hits with amorphous PET
  film/powder; quantify released **MHET / TPA** by HPLC over time and **temperature** (30 / 50 / 65 °C)
  — the real plastic-degradation readout, and where thermostability pays off.
- **Readout note:** subtract buffer-only background; for PET, surface area / crystallinity strongly
  affect rates — standardise the substrate.

## 3 · DSF thermostability + the mandatory controls
The whole project is optimised for thermostability, so **measure it**: differential scanning
fluorimetry (DSF / thermal shift) → report **Tm**, compared to a natural reference. Then the controls
every plate needs.

In [ ]:
controls = {
    "POSITIVE — natural/reference PET hydrolase": "a verified IsPETase / LCC / thermostable cutinase; "
        "confirms the assay works and benchmarks activity + Tm",
    "NEGATIVE — catalytic Ser -> Ala 'dead' mutant": "SAME design, catalytic Ser mutated to Ala; "
        "cleanest negative — loss of activity pins catalysis to the nucleophile (His/Asp inactive without it)",
    "NEGATIVE — heat-killed enzyme": "boiled aliquot; rules out non-protein / contaminant ester hydrolysis",
    "NEGATIVE — empty-vector lysate": "no insert; rules out host-background esterase activity",
    "BLANK — buffer + substrate only": "the uncatalysed background ester hydrolysis to subtract",
}
print("MANDATORY controls (every plate):")
for k, v in controls.items():
    print(f"  - {k}\n      {v}")
print("\nDSF: run thermal melts on each design + the natural reference; report Tm (deg C). This is the")
print("measurement the campaign's thermostability ranking is trying to predict — close the loop here.")

## 4 · A costed, plate-based screen (template — fill real prices)
One 96-well plate holds the < 96 designs + the controls above. Cost the gene synthesis, expression,
the pNP-ester + PET-film/HPLC assays, and DSF at your institution's rates; the cell prints a template
to fill in your report.

In [ ]:
plan = [
    ("Gene synthesis (codon-optimised, screened provider)", "< 96 designs", "fill price/construct"),
    ("Cloning + transformation", "1 plate", "fill"),
    ("Expression + lysis", "1 plate", "fill"),
    ("IMAC purification (plate format)", "1 plate", "fill"),
    ("pNP-ester substrate (pNP-acetate/butyrate)", "stock", "fill"),
    ("Amorphous PET film/powder + HPLC consumables (MHET/TPA)", "per assay", "fill"),
    ("DSF / thermal-shift reagents + instrument time", "per plate", "fill"),
    ("Plate-reader time (pNP kinetics)", "per plate", "fill"),
]
print("Costed reagent/step list (fill institutional prices) [TEMPLATE]:")
for step, scale, cost in plan:
    print(f"  - {step:54s} {scale:12s} {cost}")
print("\nTimeline (typical): synthesis 2-3 wk -> clone/express 1-2 wk -> purify+pNP screen 1 wk ->")
print("                    DSF + PET-film/HPLC on hits 1-2 wk.")
print("Synthesis MUST go through an IGSC-member, biosecurity-screening provider (low dual-use here —")
print("industrial/green-chemistry — but it is policy). Wet-lab needs institutional biosafety/ethics sign-off.")

## 5 · Surface-residue redesign for solubility `[stretch]`
A thermostable hit that won't express solubly is useless. If a top design is poorly soluble, redesign
**only the surface** residues (LigandMPNN / ProteinMPNN with the **triad + core fixed**) to improve
solubility/expression — then **re-check the catalytic geometry and the thermostability proxy**, since
surface changes can shift both. (Optionally, sketch a directed-evolution loop for activity using the
pNP screen as the readout — the honest history is that designs often need it.)

In [ ]:
print("Surface-redesign-for-solubility loop (stretch):")
print("  pick a thermostable hit with poor predicted solubility ->")
print("  LigandMPNN/ProteinMPNN redesign SURFACE only (triad + oxyanion hole + core FIXED) ->")
print("  re-predict (AF2) -> re-check catalytic_geometry_rmsd AND the thermostability proxy ->")
print("  keep variants that improve solubility WITHOUT breaking geometry/stability.")
print("\nReminder for the thesis: report the hit rate honestly, and that a thermostable, well-folded")
print("design can still be catalytically inactive (geometry != activity; MD proxy != Tm). The assay decides.")

## D4 / D5 checklist
- [ ] `results/synthesis_set.csv`: < 96 diverse, catalytic-geometry-passing, thermostability-ranked designs.
- [ ] Activity-assay plan: pNP-ester fast screen + PET-film/HPLC true substrate, initial rates / kinetics.
- [ ] **DSF thermostability** plan (report Tm vs a natural reference) — close the loop on the ranking.
- [ ] **All** controls (catalytic-Ser→Ala dead mutant, heat-killed, empty vector, blank), costed + timed.
- [ ] Surface-redesign-for-solubility (and optional directed-evolution) plan for hits `[stretch]`.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release; "geometry ≠ activity" + "MD proxy ≠ Tm" stated plainly.

You're done — this project followed the Project 18 theozyme→scaffold→sequence→geometry template, with ester hydrolysis + a **thermostability** emphasis.